In [2]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

tesla_text = """Tesla's Q3 Results
Tesla reported record revenue of $25.2B in Q3 2024.
The company exceeded analyst expectations by 15%.
Revenue growth was driven by strong vehicle deliveries.

Model Y Performance  
The Model Y became the best-selling vehicle globally, with 350,000 units sold.
Customer satisfaction ratings reached an all-time high of 96%.
Model Y now represents 60% of Tesla's total vehicle sales.

Production Challenges
Supply chain issues caused a 12% increase in production costs.
Tesla is working to diversify its supplier base.
New manufacturing techniques are being implemented to reduce costs."""


**Basic Chunking Method With CharacterTextSplitter**
- This is the most basic approach. It splits text based on a fixed number of characters or a single specific character (like a space or a comma)
- It counts characters until it hits your chunk_size limit and cuts the text right there.
- It is "blind" to context. It might cut a word in half or separate a subject from its verb, making the chunk meaningless for the AI.
- If wasn't find the separator, you will get the larger chunk size than what you define.

In [7]:
splitter1 = CharacterTextSplitter(
    separator=" ",  # Default separator. Other options include ["\n\n", "\n", ". ", " ", ""]
    chunk_size=100,
    chunk_overlap=10
)

chunks1 = splitter1.split_text(tesla_text)
for i, chunk in enumerate(chunks1, 1):
    print(f"Chunk {i}: ({len(chunk)} chars)")
    print(f'"{chunk}"')
    print()

Chunk 1: (99 chars)
"Tesla's Q3 Results
Tesla reported record revenue of $25.2B in Q3 2024.
The company exceeded analyst"

Chunk 2: (93 chars)
"analyst expectations by 15%.
Revenue growth was driven by strong vehicle deliveries.

Model Y"

Chunk 3: (87 chars)
"Y Performance 
The Model Y became the best-selling vehicle globally, with 350,000 units"

Chunk 4: (97 chars)
"units sold.
Customer satisfaction ratings reached an all-time high of 96%.
Model Y now represents"

Chunk 5: (98 chars)
"represents 60% of Tesla's total vehicle sales.

Production Challenges
Supply chain issues caused a"

Chunk 6: (95 chars)
"caused a 12% increase in production costs.
Tesla is working to diversify its supplier base.
New"

Chunk 7: (73 chars)
"base.
New manufacturing techniques are being implemented to reduce costs."



**Recursive Character Text Splitter**
- This is the industry standard and the default for most LangChain projects.
- It uses a hierarchy of separators—usually ["\n\n", "\n", " ", ""]. It tries to split by double newlines (paragraphs) first. If a paragraph is still too big, it tries single newlines, then spaces, and finally individual characters.
- It keeps related text together (like sentences and paragraphs) as much as possible while strictly staying under your size limit.

In [8]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],  # Multiple separators
    chunk_size=100,
    chunk_overlap=0
)

chunks2 = recursive_splitter.split_text(tesla_text)
print(f"Same problem text, but with RecursiveCharacterTextSplitter:")
for i, chunk in enumerate(chunks2, 1):
    print(f"Chunk {i}: ({len(chunk)} chars)")
    print(f'"{chunk}"')
    print()

Same problem text, but with RecursiveCharacterTextSplitter:
Chunk 1: (70 chars)
"Tesla's Q3 Results
Tesla reported record revenue of $25.2B in Q3 2024."

Chunk 2: (49 chars)
"The company exceeded analyst expectations by 15%."

Chunk 3: (55 chars)
"Revenue growth was driven by strong vehicle deliveries."

Chunk 4: (19 chars)
"Model Y Performance"

Chunk 5: (78 chars)
"The Model Y became the best-selling vehicle globally, with 350,000 units sold."

Chunk 6: (62 chars)
"Customer satisfaction ratings reached an all-time high of 96%."

Chunk 7: (58 chars)
"Model Y now represents 60% of Tesla's total vehicle sales."

Chunk 8: (84 chars)
"Production Challenges
Supply chain issues caused a 12% increase in production costs."

Chunk 9: (48 chars)
"Tesla is working to diversify its supplier base."

Chunk 10: (67 chars)
"New manufacturing techniques are being implemented to reduce costs."



**Semantic Chunking**
- This is the "intelligent" approach that uses Machine Learning to determine where to cut.
- Instead of counting characters, it looks at the meaning of the text. It calculates the embedding (vector) for each sentence. It then looks for "breakpoints" where the meaning of one sentence is significantly different from the previous one.
- Chunks are grouped by topic. This ensures that a single chunk contains a complete idea, leading to much higher retrieval accuracy.

In [9]:
from langchain_ollama import OllamaEmbeddings

semantic_splitter = SemanticChunker(
    embeddings=OllamaEmbeddings(model="nomic-embed-text"),
    breakpoint_threshold_type="percentile",  # or "standard_deviation"
    breakpoint_threshold_amount=70
)

chunks = semantic_splitter.split_text(tesla_text)

print("SEMANTIC CHUNKING RESULTS:")
print("=" * 50)
for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}: ({len(chunk)} chars)")
    print(f'"{chunk}"')
    print()

SEMANTIC CHUNKING RESULTS:
Chunk 1: (120 chars)
"Tesla's Q3 Results
Tesla reported record revenue of $25.2B in Q3 2024. The company exceeded analyst expectations by 15%."

Chunk 2: (219 chars)
"Revenue growth was driven by strong vehicle deliveries. Model Y Performance  
The Model Y became the best-selling vehicle globally, with 350,000 units sold. Customer satisfaction ratings reached an all-time high of 96%."

Chunk 3: (58 chars)
"Model Y now represents 60% of Tesla's total vehicle sales."

Chunk 4: (201 chars)
"Production Challenges
Supply chain issues caused a 12% increase in production costs. Tesla is working to diversify its supplier base. New manufacturing techniques are being implemented to reduce costs."

